In [12]:
import torch

from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from transformers import Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from PIL import Image

if not hasattr(torch.compiler, "is_compiling"):
    torch.compiler.is_compiling = lambda: False

In [11]:
# !pip install --upgrade transformers
# !pip install --upgrade "huggingface-hub>=0.34.0,<1.0"
# !pip install qwen-vl-utils

### Qwen2.5-VL-7B-Instruct

In [1]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct"
)

In [2]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

In [7]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "/home/jovyan/nkiselev/ddorin/project/Image-Transform-Predict/assets/image.jpg"},
            {"type": "image", "image": "/home/jovyan/nkiselev/ddorin/project/Image-Transform-Predict/assets/transformed_image.jpg"},
            {"type": "text", "text": "Identify the similarities between these images."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['These two images share the following similarities:\n\n1. **Subject**: Both images feature animals, specifically dogs.\n2. **Outdoor Setting**: Both photos appear to be taken outdoors, with greenery in the background.\n3. **Focus on the Animal**: The primary focus of both images is on the animal, with the background blurred to emphasize the subject.\n4. **Natural Lighting**: Both images seem to have been captured using natural light, giving them a soft and warm tone.\n\nThe differences lie in the specific breeds, colors, and poses of the dogs, as well as the overall mood and composition of each photo.']


In [12]:
prompt = """You are given two images: Image A (original) and Image B (transformed).  
Your task is to predict the sequence of transformations applied to Image A to obtain Image B, using **only** the following allowed operations:  
"noop", "grayscale", "rotate_90", "rotate_180", "rotate_270", "color_jitter", "noise_adding", "crop", "horizontal_flip", "vertical_flip".

- The sequence may contain **zero, one, or multiple** transformations applied in order.  
- If Image A and Image B are identical, return: ["noop"]  
- If Image B can be obtained by applying a sequence of the allowed transformations (in the correct order), return that sequence as a JSON list, e.g.: ["color_jitter", "noise_adding", "rotate_270", "horizontal_flip"]  
- If the transformation from Image A to Image B **requires any operation not in the allowed list** (e.g., blur, resize, perspective distortion, custom warping, etc.), or if the images are unrelated, return an empty list: []  

Output only the JSON list. Do not add explanations, comments, or extra text."""

In [15]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "/home/jovyan/nkiselev/ddorin/project/Image-Transform-Predict/assets/dog.jpg"},
            {"type": "image", "image": "/home/jovyan/nkiselev/ddorin/project/Image-Transform-Predict/assets/transformed_dog.jpg"},
            {"type": "text", "text": prompt},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['["color_jitter"]']


### Qwen3-VL-4B-Instruct

In [7]:
# !pip install transformers==4.57.0
# https://github.com/QwenLM/Qwen3-VL/blob/main/qwen-vl-utils/src/qwen_vl_utils/vision_process.py
# https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct/blob/main/config.json

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
)

# Store processor for preprocessing
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct", )

# vision_model = model.visual

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
model.to('cuda')

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [15]:
# наша красотка занимает 17362MiB / 24576MiB

In [16]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "/mnt/DATA2/dorin/imgs/car/image1.jpg"},
            {"type": "image", "image": "/mnt/DATA2/dorin/imgs/car/image2.jpg"},
            {"type": "text", "text": "Identify the similarities between these images."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['Based on a careful comparison of the two images, here are the key similarities:\n\n- **Subject:** Both images feature the exact same silver sedan, which is identifiable as a BMW 3 Series (likely a 330i or similar model from the F30 generation) due to its distinctive body lines, kidney grille, and wheel design.\n- **Angle and Composition:** The car is captured from the exact same side profile angle in both photos. The framing, including the position of the car relative to the foreground and background, is identical.\n- **Position and Environment:** The car is parked in the same location, on a gravel']


In [10]:
image_path = 'assets/image.jpg'

image = Image.open(image_path)
out = processor.image_processor(image)

print(out['pixel_values'].shape, out['image_grid_thw'])

torch.Size([2604, 1536]) tensor([[ 1, 42, 62]])
